In [32]:
import pandas as pd
import numpy as np

train = pd.read_parquet("train.parquet")
inference = pd.read_parquet("inference.parquet")
capacity = pd.read_parquet("capacity.parquet")

print("Data loaded successfully!")


Data loaded successfully!


In [33]:
print("Missing values in each column:")
print(train.isnull().sum())

print("\nNumber of duplicate rows:")
print(train.duplicated().sum())

print("\n First five rows:")
print(train.head())

print("\nColumn data types:")
print(train.dtypes)


Missing values in each column:
sales_order_id        0
buid                  0
site                  0
orig_wo_id            0
tie_num               0
mcid                  0
order_amt             0
quantity_produced     0
line_qty              0
cfs_flag              0
is_cfi                0
alt_type_desc         0
lob_desc              0
family_parent_desc    0
family_desc           0
download_date         5
kit_start             0
kit_end               0
est_ship_date         0
dtype: int64

Number of duplicate rows:
0

 First five rows:
   sales_order_id buid site   orig_wo_id tie_num mcid    order_amt  \
0      1025548771   11  KHO  22067915245       1  3JU  42935.31840   
1      1027927868   11  KHO  22095717945       1  KHO  33128.74827   
2      1029939204   11  KHO  22112237255       1  KHO  17238.22614   
3      1023379512   11  KHO  22082951775       1  3JU  34306.97076   
4      1017735495  707  KHO  21983065475       1  KHO   6261.03846   

  quantity_produced line_qty  c

In [34]:
train["kit_duration"] = (train["kit_end"] - train["kit_start"]).dt.days

print("Average days:", train["kit_duration"].mean())
print("Shortest duration:", train["kit_duration"].min())
print("Longest duration:", train["kit_duration"].max())

negative = train[train["kit_duration"] < 0]
print("Negative durations:", len(negative))

Average days: 1.3385431435672466
Shortest duration: -25
Longest duration: 76
Negative durations: 734


In [35]:
# Find orders with negative durations
negative = train[train["kit_duration"] < 0]

# Show the first 10 unusual orders
print(negative[["sales_order_id", "kit_start", "kit_end", "kit_duration"]].head(10))

      sales_order_id  kit_start    kit_end  kit_duration
465       1020850367 2025-07-08 2025-06-27           -11
873       1020912983 2025-06-27 2025-06-26            -1
953       1026592472 2025-11-14 2025-11-12            -2
1244      1020961824 2025-07-08 2025-06-27           -11
1493      1020963295 2025-07-08 2025-06-27           -11
1663      1020933864 2025-07-08 2025-06-26           -12
2056      1020301692 2025-07-08 2025-06-27           -11
2245      1020950271 2025-07-08 2025-06-26           -12
2280      1020950529 2025-07-08 2025-06-27           -11
2496      1020946449 2025-07-08 2025-06-26           -12


In [36]:
# Count orders that finish before they start
print("Negative durations:", (train["kit_duration"] < 0).sum())

# Count orders that finish on the same day they start
print("Zero-day durations:", (train["kit_duration"] == 0).sum())

# Count orders that finish after they start
print("Positive durations:", (train["kit_duration"] > 0).sum())

Negative durations: 734
Zero-day durations: 35082
Positive durations: 144037


In [37]:
# Show the columns available for prediction
print("Inference columns:")
print(inference.columns.tolist())

# Check how many rows need predictions
print("\nNumber of orders to predict:")
print(len(inference))

Inference columns:
['sales_order_id', 'buid', 'site', 'orig_wo_id', 'tie_num', 'mcid', 'order_amt', 'quantity_produced', 'line_qty', 'cfs_flag', 'is_cfi', 'alt_type_desc', 'lob_desc', 'family_parent_desc', 'family_desc', 'download_date', 'kit_start', 'est_ship_date']

Number of orders to predict:
12069


In [38]:
# Calculate how many days each order takes
train["kit_duration"] = (train["kit_end"] - train["kit_start"]).dt.days

# Make a new dataset without negative durations
train_clean = train[train["kit_duration"] >= 0]

# Print the number of rows
print("Original rows:", len(train))
print("Rows after removing negative durations:", len(train_clean))


Original rows: 179853
Rows after removing negative durations: 179119


In [39]:
# Choose info that might help predict the duration
features = [
    "buid",
    "site",
    "mcid",
    "order_amt",
    "quantity_produced",
    "line_qty",
    "cfs_flag",
    "is_cfi",
    "download_date",
    "kit_start",
    "est_ship_date"
]

# Show the info we selected
print("Number of input columns:", len(features))
print(features)


Number of input columns: 11
['buid', 'site', 'mcid', 'order_amt', 'quantity_produced', 'line_qty', 'cfs_flag', 'is_cfi', 'download_date', 'kit_start', 'est_ship_date']


In [40]:
# Create year, month, and day features from the dates
train_clean["download_year"] = train_clean["download_date"].dt.year
train_clean["download_month"] = train_clean["download_date"].dt.month

train_clean["kit_start_month"] = train_clean["kit_start"].dt.month
train_clean["kit_start_day"] = train_clean["kit_start"].dt.day

train_clean["ship_month"] = train_clean["est_ship_date"].dt.month
train_clean["ship_day"] = train_clean["est_ship_date"].dt.day

# Show the new columns
print(train_clean[[
    "download_year",
    "download_month",
    "kit_start_month",
    "kit_start_day",
    "ship_month",
    "ship_day"
]].head())


   download_year  download_month  kit_start_month  kit_start_day  ship_month  \
0         2025.0            10.0               10             16          10   
1         2025.0            12.0               12             12           3   
2         2026.0             1.0                1             30           6   
3         2025.0            11.0               11             29          11   
4         2025.0             4.0                4              3           4   

   ship_day  
0        31  
1        16  
2        17  
3        12  
4        16  


In [41]:
# Choose the columns the model will learn from
features = [
    "buid",
    "site",
    "mcid",
    "order_amt",
    "quantity_produced",
    "line_qty",
    "cfs_flag",
    "is_cfi",
    "download_year",
    "download_month",
    "kit_start_month",
    "kit_start_day",
    "ship_month",
    "ship_day"
]

# Save the input data
in_data = train_clean[features]

# Save the answer the model will learn to predict
in_data_ans = train_clean["kit_duration"]

# Convert text categories into numbers
in_data = pd.get_dummies(in_data)

# Fill the missing numeric values with the median
in_data = in_data.fillna(in_data.median())

# Check our results
print("Number of training rows:", len(in_data))
print("Number of input columns:", len(in_data.columns))
print("Missing values:", in_data.isnull().sum().sum())


Number of training rows: 179119
Number of input columns: 68
Missing values: 0


In [42]:
from sklearn.model_selection import train_test_split

# Split the data into training and testing parts
X_train, X_test, y_train, y_test = train_test_split(
    in_data, in_data_ans, test_size=0.2, random_state=42
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))


Training rows: 143295
Testing rows: 35824


In [43]:
from sklearn.ensemble import RandomForestRegressor

# Create the model
model = RandomForestRegressor(
    n_estimators = 100,
    random_state = 42,
    n_jobs =- 1
)

# Teach the model using the training data
model.fit(X_train, y_train)

# Show a message when training is finished
print("Model training complete!")

Model training complete!


In [44]:
# Make predictions using the test data
predictions = model.predict(X_test)

# Calculate the average error
from sklearn.metrics import mean_absolute_error

error = mean_absolute_error(y_test, predictions)

# Show the result
print("Average prediction error:", error, "days")


Average prediction error: 0.5219799012437439 days


In [45]:
# Calculate the average duration in the training data
average_days = y_train.mean()

# Compare every test order against that average
simple_predictions = [average_days] * len(y_test)

# Calculate the error of the simple prediction
simple_error = mean_absolute_error(y_test, simple_predictions)

print("Random Forest error:", error, "days")
print("Simple prediction error:", simple_error, "days")

Random Forest error: 0.5219799012437439 days
Simple prediction error: 0.9129007453087505 days


In [46]:
# Create date columns
inference["download_year"] = inference["download_date"].dt.year
inference["download_month"] = inference["download_date"].dt.month
inference["kit_start_month"] = inference["kit_start"].dt.month
inference["kit_start_day"] = inference["kit_start"].dt.day
inference["ship_month"] = inference["est_ship_date"].dt.month
inference["ship_day"] = inference["est_ship_date"].dt.day

# Select the same features used for training
new_data = inference[features]

# Turn text columns into numbers
new_data = pd.get_dummies(new_data)

# Match the training columns
new_data = new_data.reindex(columns=in_data.columns, fill_value=0)

# Fill missing values
new_data = new_data.fillna(in_data.median())

# Check the data
print("Number of rows:", len(new_data))
print("Number of columns:", len(new_data.columns))
print("Missing values:", new_data.isna().sum().sum())


Number of rows: 12069
Number of columns: 68
Missing values: 0


In [47]:
# Make predictions
predictions = model.predict(new_data)

# Round the predictions
predictions = np.round(predictions)

# Add the predicted days to the start date
kit_end_predictions = inference["kit_start"] + pd.to_timedelta(predictions, unit="D")

# Show the first 10 predictions
print(kit_end_predictions.head(10))

# Count the predictions
print("Number of predictions:", len(kit_end_predictions))

0   2026-02-14
1   2026-02-12
2   2026-02-27
3   2026-02-13
4   2026-02-13
5   2026-02-12
6   2026-03-08
7   2026-03-06
8   2026-02-26
9   2026-02-05
dtype: datetime64[ns]
Number of predictions: 12069


In [48]:
# Save the predictions
np.save("predictions.npy", kit_end_predictions.to_numpy())

print("Predictions saved!")
print("Total predictions:", len(kit_end_predictions))


Predictions saved!
Total predictions: 12069


In [49]:
# Load the saved predictions
saved_predictions = np.load("predictions.npy")

# Check the file
print("File loaded successfully!")
print("Number of predictions:", len(saved_predictions))
print("First 5 predictions:", saved_predictions[:5])

File loaded successfully!
Number of predictions: 12069
First 5 predictions: ['2026-02-14T00:00:00.000000000' '2026-02-12T00:00:00.000000000'
 '2026-02-27T00:00:00.000000000' '2026-02-13T00:00:00.000000000'
 '2026-02-13T00:00:00.000000000']


In [50]:
import numpy as np

saved_prediction = np.load("predictions.npy")

print("Number of predictions:", len(saved_predictions))
print("First 5 predictions:", saved_predictions[:5])

Number of predictions: 12069
First 5 predictions: ['2026-02-14T00:00:00.000000000' '2026-02-12T00:00:00.000000000'
 '2026-02-27T00:00:00.000000000' '2026-02-13T00:00:00.000000000'
 '2026-02-13T00:00:00.000000000']


In [51]:
# Sort orders by start date
train_sorted = train_clean.sort_values("kit_start")

# Split the data: first 80% for training, last 20% for testing
split_point = int(len(train_sorted) * 0.8)

train_part = train_sorted.iloc[:split_point]
test_part = train_sorted.iloc[split_point:]

# Choose the features and answers
X_train_time = train_part[features]
y_train_time = train_part["kit_duration"]

X_test_time = test_part[features]
y_test_time = test_part["kit_duration"]

# Convert categories into numbers
X_train_time = pd.get_dummies(X_train_time)
X_test_time = pd.get_dummies(X_test_time)

# Make the columns match
X_test_time = X_test_time.reindex(
    columns=X_train_time.columns, fill_value=0
)

# Fill missing values
X_train_time = X_train_time.fillna(X_train_time.median())
X_test_time = X_test_time.fillna(X_train_time.median())

print("Training rows:", len(X_train_time))
print("Testing rows:", len(X_test_time))

Training rows: 143295
Testing rows: 35824


In [52]:
# Create the model
time_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

# Train the model
time_model.fit(X_train_time, y_train_time)

# Predict the test durations
time_predictions = time_model.predict(X_test_time)

# Calculate the prediction error
time_error = mean_absolute_error(y_test_time, time_predictions)

print("Time-based prediction error:", time_error, "days")

Time-based prediction error: 1.6050692515845846 days


In [53]:
print("Training start date:", train_part["kit_start"].min())
print("Training end date:", train_part["kit_start"].max())

print("Testing start date:", test_part["kit_start"].min())
print("Testing end date:", test_part["kit_start"].max())

Training start date: 2025-03-04 00:00:00
Training end date: 2025-11-22 00:00:00
Testing start date: 2025-11-22 00:00:00
Testing end date: 2026-01-31 00:00:00


In [54]:
# Find the date for the split
split_date = train_sorted["kit_start"].quantile(0.8)

# Split the orders by date
train_part = train_sorted[train_sorted["kit_start"] < split_date]
test_part = train_sorted[train_sorted["kit_start"] >= split_date]

# Show the results
print("Training ends:", train_part["kit_start"].max())
print("Testing starts:", test_part["kit_start"].min())

print("Training rows:", len(train_part))
print("Testing rows:", len(test_part))

Training ends: 2025-11-21 00:00:00
Testing starts: 2025-11-22 00:00:00
Training rows: 142675
Testing rows: 36444


In [55]:
# Choose the training data
X_train_time = train_part[features]
y_train_time = train_part["kit_duration"]

# Choose the testing data
X_test_time = test_part[features]
y_test_time = test_part["kit_duration"]

# Convert text columns into numbers
X_train_time = pd.get_dummies(X_train_time)
X_test_time = pd.get_dummies(X_test_time)

# Make the testing columns match the training columns
X_test_time = X_test_time.reindex(columns=X_train_time.columns, fill_value=0)

# Fill missing values with the median
X_train_time = X_train_time.fillna(X_train_time.median())
X_test_time = X_test_time.fillna(X_train_time.median())

# Check the results
print("Training rows:", len(X_train_time))
print("Testing rows:", len(X_test_time))

Training rows: 142675
Testing rows: 36444


In [56]:
# Create the model
time_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

# Train the model
time_model.fit(X_train_time, y_train_time)

# Make predictions for the testing data
time_predictions = time_model.predict(X_test_time)

# Calculate the prediction error
time_mae = mean_absolute_error(y_test_time, time_predictions)

print("Time-based MAE:", time_mae)

Time-based MAE: 1.54024316994209


In [57]:
# Predict the average duration for every test order
baseline_predictions = [y_train_time.mean()] * len(y_test_time)

# Calculate the baseline error
baseline_mae = mean_absolute_error(
    y_test_time, baseline_predictions
)

print("Time-based model MAE:", time_mae)
print("Time-based baseline MAE:", baseline_mae)

Time-based model MAE: 1.54024316994209
Time-based baseline MAE: 1.0034372542201273


In [58]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error

# Create the decision tree model
tree_model = DecisionTreeRegressor(
    random_state=42,
    max_depth=10
)

# Train the model
tree_model.fit(X_train_time, y_train_time)

# Predict the testing data
tree_predictions = tree_model.predict(X_test_time)

# Calculate the prediction error
tree_mae = mean_absolute_error(
    y_test_time, tree_predictions
)

# Print the model results
print("Average baseline MAE:", baseline_mae)
print("Random Forest MAE:", time_mae)
print("Decision Tree MAE:", tree_mae)


Average baseline MAE: 1.0034372542201273
Random Forest MAE: 1.54024316994209
Decision Tree MAE: 1.5444984909117558


In [59]:
# Save the average duration for each site
site_averages = {}

for site in train_part["site"].unique():
    total = 0
    count = 0

    for i in range(len(train_part)):
        if train_part.iloc[i]["site"] == site:
            total += train_part.iloc[i]["kit_duration"]
            count += 1

    site_averages[site] = total / count

# Make predictions for the testing data
site_predictions = []

for i in range(len(test_part)):
    site = test_part.iloc[i]["site"]

    if site in site_averages:
        site_predictions.append(site_averages[site])
    else:
        site_predictions.append(y_train_time.mean())

# Measure the prediction error
site_mae = mean_absolute_error(
    y_test_time, site_predictions
)

print("Site-based MAE:", site_mae)
print("Overall baseline MAE:", baseline_mae)


Site-based MAE: 1.0034372542201273
Overall baseline MAE: 1.0034372542201273


In [60]:
# Keep historical orders with valid kit durations
valid_orders = train_clean[train_clean["kit_duration"] >= 0]

# Calculate the average kit duration
average_duration = valid_orders["kit_duration"].mean()

# Predict the kit end dates
predicted_end_dates = (
    inference["kit_start"].dt.normalize()
    + pd.to_timedelta(round(average_duration), unit="D")
)

# Save the predictions
np.save("predictions.npy", predicted_end_dates.to_numpy())

# Check the results
print("Average kit duration:", average_duration)
print("Number of predictions:", len(predicted_end_dates))
print(predicted_end_dates.head())

Average kit duration: 1.3856263154662543
Number of predictions: 12069
0   2026-02-14
1   2026-02-11
2   2026-02-27
3   2026-02-13
4   2026-02-12
Name: kit_start, dtype: datetime64[ns]


In [61]:
# Load the saved predictions
saved_predictions = np.load("predictions.npy")

# Check the results
print("Number of saved predictions:", len(saved_predictions))
print("First five saved predictions:")
print(saved_predictions[:5])

Number of saved predictions: 12069
First five saved predictions:
['2026-02-14T00:00:00.000000000' '2026-02-11T00:00:00.000000000'
 '2026-02-27T00:00:00.000000000' '2026-02-13T00:00:00.000000000'
 '2026-02-12T00:00:00.000000000']


In [ ]:
# Load the saved predictions
saved_predictions = np.load("predictions.npy")

# Check the number of inference orders and predictions
print("Inference orders:", len(inference))
print("Saved predictions:", len(saved_predictions))

# Check the first few start dates and predicted end dates
print(inference["kit_start"].head())
print(saved_predictions[:5])

Inference orders: 12069
Saved predictions: 12069
0   2026-02-13
1   2026-02-10
2   2026-02-26
3   2026-02-12
4   2026-02-11
Name: kit_start, dtype: datetime64[ns]
['2026-02-14T00:00:00.000000000' '2026-02-11T00:00:00.000000000'
 '2026-02-27T00:00:00.000000000' '2026-02-13T00:00:00.000000000'
 '2026-02-12T00:00:00.000000000']


In [63]:
print("Prediction array shape:", saved_predictions.shape)
print("Prediction data type:", saved_predictions.dtype)
print("First prediction:", saved_predictions[0])
print("Last prediction:", saved_predictions[-1])

Prediction array shape: (12069,)
Prediction data type: datetime64[ns]
First prediction: 2026-02-14T00:00:00.000000000
Last prediction: 2026-02-04T00:00:00.000000000
